# Hybrid Router Rules v0/v1

Notebook ??? ???????? ??????? production-safe ?????? ????????????? ??????? ?? SKU.

???????????? ?????:
- `single_model_66`: ???? ??????? ?????? ?? ??? SKU;
- `oracle_best`: ?????? ?????? ?? ????????????? benchmark-?;
- `rule_based_v0`: ?????? ?????? ??????;
- `rule_based_v1`: ????? ?????? ?????? ??????.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.experiments_v2.benchmark_common import BAKERY_COL, DATE_COL, DEMAND_TARGET, PRODUCT_COL, load_daily_demand
from src.experiments_v2.monthly_benchmark_common import build_exp63_feature_frame

FULL_BENCHMARK_PATH = ROOT / 'src' / 'experiments_v2' / 'Full_benchmark_mounth_results' / 'src' / 'experiments_v2' / 'full_benchmark_monthly' / 'best_by_sku.csv'
SKU_LOCAL_PATH = ROOT / 'src' / 'experiments_v2' / 'Full_benchmark_mounth_results' / 'src' / 'experiments_v2' / 'sku_local_monthly' / 'summary_best_by_sku.csv'
ARTIFACT_DIR = ROOT / 'reports' / 'hybrid_research'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
sns.set_theme(style='whitegrid')

print('ROOT:', ROOT)
print('FULL_BENCHMARK_PATH:', FULL_BENCHMARK_PATH)
print('SKU_LOCAL_PATH:', SKU_LOCAL_PATH)
print('ARTIFACT_DIR:', ARTIFACT_DIR)


In [ ]:
full_df = pd.read_csv(FULL_BENCHMARK_PATH, encoding='utf-8-sig')
sku_local_df = pd.read_csv(SKU_LOCAL_PATH, encoding='utf-8-sig')

benchmark = full_df.merge(
    sku_local_df,
    on=[BAKERY_COL, PRODUCT_COL],
    how='inner',
    suffixes=('_full', '_local'),
    validate='one_to_one',
)

raw_df = load_daily_demand()
featured_df = build_exp63_feature_frame(raw_df)

sku_features = (
    featured_df.groupby([BAKERY_COL, PRODUCT_COL], sort=False)
    .agg(
        n_days=(DATE_COL, 'nunique'),
        mean_demand=(DEMAND_TARGET, 'mean'),
        median_demand=(DEMAND_TARGET, 'median'),
        std_demand=(DEMAND_TARGET, 'std'),
        share_zero=(DEMAND_TARGET, lambda s: float((s == 0).mean())),
        share_censored=('is_censored', 'mean'),
        lost_mean=('lost_qty', 'mean'),
        demand_trend_mean=('demand_trend', 'mean'),
        cv_7d_mean=('cv_7d', 'mean'),
    )
    .reset_index()
)
sku_features['cv_demand'] = sku_features['std_demand'].fillna(0) / (sku_features['mean_demand'].abs() + 1e-8)

router_df = benchmark.merge(sku_features, on=[BAKERY_COL, PRODUCT_COL], how='left', validate='one_to_one')
model_r2_cols = [c for c in router_df.columns if c.endswith('_r2')]
router_df['has_model_metrics'] = router_df[model_r2_cols].notna().any(axis=1)
router_df['all_model_metrics_missing'] = router_df[model_r2_cols].isna().all(axis=1)
router_df['is_short_history'] = router_df['n_days'].fillna(0) <= 1

router_df_clean = router_df.loc[router_df['has_model_metrics'] & (~router_df['is_short_history'])].copy()
router_df_fallback = router_df.loc[(~router_df['has_model_metrics']) | (router_df['is_short_history'])].copy()

print('rows total:', len(router_df))
print('rows clean:', len(router_df_clean))
print('rows fallback:', len(router_df_fallback))
print('missing model metrics:', int(router_df['all_model_metrics_missing'].sum()))
print('short history:', int(router_df['is_short_history'].sum()))
display(router_df_clean.head(3))

router_df.to_csv(ARTIFACT_DIR / 'router_candidates.csv', index=False, encoding='utf-8-sig')
router_df_clean.to_csv(ARTIFACT_DIR / 'router_candidates_clean.csv', index=False, encoding='utf-8-sig')
router_df_fallback.to_csv(ARTIFACT_DIR / 'router_candidates_fallback.csv', index=False, encoding='utf-8-sig')


In [ ]:
candidate_models = [
    '60_baseline_v3',
    '61_censoring_behavioral',
    '62_assortment_availability',
    '66_cluster_features',
    'prophet_local',
    'lgbm_local',
    'two_week_avg',
]

def metric_col(model: str, metric: str) -> str:
    return f'{model}_{metric}'

available_models = [m for m in candidate_models if metric_col(m, 'r2') in router_df_clean.columns]
print('available models:', available_models)

r2_cols = [metric_col(m, 'r2') for m in available_models]
r2_values = router_df_clean[r2_cols].apply(pd.to_numeric, errors='coerce')
best_idx = r2_values.idxmax(axis=1)

oracle = router_df_clean[[BAKERY_COL, PRODUCT_COL]].copy()
oracle['oracle_model'] = best_idx.str.replace('_r2', '', regex=False)
oracle['oracle_r2'] = r2_values.max(axis=1).to_numpy()
oracle['oracle_mae'] = [router_df_clean.iloc[i][metric_col(oracle.iloc[i]['oracle_model'], 'mae')] for i in range(len(router_df_clean))]
oracle['oracle_mse'] = [router_df_clean.iloc[i][metric_col(oracle.iloc[i]['oracle_model'], 'mse')] for i in range(len(router_df_clean))]
oracle['oracle_wmape'] = [router_df_clean.iloc[i][metric_col(oracle.iloc[i]['oracle_model'], 'wmape')] for i in range(len(router_df_clean))]

display(oracle.head(3))
oracle.to_csv(ARTIFACT_DIR / 'oracle_best_by_sku.csv', index=False, encoding='utf-8-sig')


In [ ]:
def pick_rule_based_v0(row: pd.Series) -> str:
    if row['cv_demand'] > 1.0:
        return 'two_week_avg'
    if row['mean_demand'] < 5:
        return 'two_week_avg'
    if row['share_censored'] > 0.5:
        return '61_censoring_behavioral'
    if row['cv_demand'] < 0.6 and row['mean_demand'] > 10:
        return '66_cluster_features'
    if row['share_censored'] > 0.35:
        return '62_assortment_availability'
    if row['mean_demand'] > 8 and row['cv_demand'] < 0.8:
        return 'lgbm_local'
    return '66_cluster_features'

def pick_rule_based_v1(row: pd.Series) -> str:
    if row['cv_demand'] > 1.25 and row['mean_demand'] < 8:
        return 'two_week_avg'
    if row['mean_demand'] < 4 and row['cv_demand'] > 0.8:
        return 'two_week_avg'
    if row['share_censored'] > 0.65 and row['mean_demand'] < 15:
        return '61_censoring_behavioral'
    if row['share_censored'] > 0.45 and row['mean_demand'] < 12:
        return '62_assortment_availability'
    if row['cv_demand'] < 0.7 and row['mean_demand'] > 8:
        return '66_cluster_features'
    if row['mean_demand'] > 12 and row['cv_demand'] < 0.85:
        return '66_cluster_features'
    if row['mean_demand'] > 6 and row['cv_demand'] < 1.0:
        return 'lgbm_local'
    return '66_cluster_features'

router_df_clean['rule_model_v0'] = router_df_clean.apply(pick_rule_based_v0, axis=1)
router_df_clean['rule_model_v1'] = router_df_clean.apply(pick_rule_based_v1, axis=1)
missing_rule_models_v0 = sorted(set(router_df_clean['rule_model_v0']) - set(available_models))
missing_rule_models_v1 = sorted(set(router_df_clean['rule_model_v1']) - set(available_models))
print('missing rule models v0:', missing_rule_models_v0)
print('missing rule models v1:', missing_rule_models_v1)

router_df_clean['single_model_66'] = '66_cluster_features'
router_df_clean.to_csv(ARTIFACT_DIR / 'router_candidates_with_rules.csv', index=False, encoding='utf-8-sig')
display(router_df_clean[[BAKERY_COL, PRODUCT_COL, 'mean_demand', 'cv_demand', 'share_censored', 'rule_model_v0', 'rule_model_v1']].head(10))


In [ ]:
def attach_choice_metrics(df: pd.DataFrame, choice_col: str, prefix: str) -> pd.DataFrame:
    result = df[[BAKERY_COL, PRODUCT_COL, choice_col]].copy()
    chosen = result[choice_col].astype(str)
    result[f'{prefix}_model'] = chosen
    for metric in ['r2', 'mae', 'mse', 'wmape']:
        result[f'{prefix}_{metric}'] = [df.iloc[i][metric_col(chosen.iloc[i], metric)] for i in range(len(df))]
    return result

rule_v0_metrics = attach_choice_metrics(router_df_clean, 'rule_model_v0', 'rule_v0')
rule_v1_metrics = attach_choice_metrics(router_df_clean, 'rule_model_v1', 'rule_v1')
single_metrics = attach_choice_metrics(router_df_clean, 'single_model_66', 'single')

compare_df = oracle.merge(rule_v0_metrics, on=[BAKERY_COL, PRODUCT_COL], how='left').merge(
    rule_v1_metrics, on=[BAKERY_COL, PRODUCT_COL], how='left'
).merge(
    single_metrics, on=[BAKERY_COL, PRODUCT_COL], how='left'
)

compare_df.to_csv(ARTIFACT_DIR / 'routing_comparison_rows.csv', index=False, encoding='utf-8-sig')
display(compare_df.head(3))


In [ ]:
def summarize(name: str, r2_col: str, mae_col: str, mse_col: str, wmape_col: str) -> dict:
    return {
        'scheme': name,
        'avg_r2': round(float(compare_df[r2_col].mean()), 4),
        'median_r2': round(float(compare_df[r2_col].median()), 4),
        'avg_mae': round(float(compare_df[mae_col].mean()), 4),
        'avg_mse': round(float(compare_df[mse_col].mean()), 4),
        'avg_wmape': round(float(compare_df[wmape_col].mean()), 2),
        'positive_r2_share': round(float((compare_df[r2_col] > 0).mean()), 4),
        'n_rows': int(len(compare_df)),
    }

summary = pd.DataFrame([
    summarize('oracle_best', 'oracle_r2', 'oracle_mae', 'oracle_mse', 'oracle_wmape'),
    summarize('rule_based_v0', 'rule_v0_r2', 'rule_v0_mae', 'rule_v0_mse', 'rule_v0_wmape'),
    summarize('rule_based_v1', 'rule_v1_r2', 'rule_v1_mae', 'rule_v1_mse', 'rule_v1_wmape'),
    summarize('single_model_66', 'single_r2', 'single_mae', 'single_mse', 'single_wmape'),
])

display(summary)
summary.to_csv(ARTIFACT_DIR / 'routing_summary.csv', index=False, encoding='utf-8-sig')

win_counts_v0 = compare_df['rule_v0_model'].value_counts().rename_axis('model').reset_index(name='win_count')
win_counts_v1 = compare_df['rule_v1_model'].value_counts().rename_axis('model').reset_index(name='win_count')
display(win_counts_v0)
display(win_counts_v1)
win_counts_v0.to_csv(ARTIFACT_DIR / 'routing_rule_v0_win_counts.csv', index=False, encoding='utf-8-sig')
win_counts_v1.to_csv(ARTIFACT_DIR / 'routing_rule_v1_win_counts.csv', index=False, encoding='utf-8-sig')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=summary, x='scheme', y='avg_mae', ax=axes[0], palette='viridis')
axes[0].set_title('Average MAE by scheme')
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=summary, x='scheme', y='avg_wmape', ax=axes[1], palette='magma')
axes[1].set_title('Average WMAPE by scheme')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(router_df_clean['cv_demand'], bins=60, ax=axes[0], color='#264653')
axes[0].set_title('cv_demand distribution')
sns.histplot(router_df_clean['mean_demand'], bins=60, ax=axes[1], color='#2a9d8f')
axes[1].set_title('mean_demand distribution')
plt.tight_layout()
plt.show()


## What to inspect next

- ????????? `rule_based_v1` ???????????? ? `oracle_best`;
- ???????? ?? ?? `single_model_66` ?? `MAE` ? `WMAPE`;
- ????? SKU ?? ?????????? ? `two_week_avg`, `66_cluster_features` ? censoring-??????;
- ??? rule-based routing ??????????? ? ????? ?????? ???? ?????????.
